<div style="text-align:center;">
<img src="../images/logoUTT.png" alt="UTT" style="height:90px;display:inline-block;margin:10px;" />
<img src="../images/twitter.png" alt="Twitter" style="height:75px;display:inline-block;margin:10px;" />
<img src="../images/worldcup.png" alt="FIFA World Cup 2018" style="height:100px;display:inline-block;margin:10px;" />
</div>

# IF29 : Traitement de données

## Détection de profils atypiques sur Twitter

### Comparaison d'une approche supervisée et non supervisée

**Groupe 02 - Semestre P26**  
**Dataset Tweet_Worldcup (FIFA 2018)**  
**Année universitaire 2025 - 2026**

Université de Technologie de Troyes

---



## Introduction

Twitter, devenu X depuis fin 2022, reste l'une des plateformes les plus utilisées pour partager de l'information en temps réel. Cette ouverture a aussi un revers. Des comptes automatisés, des relais de désinformation ou des profils qui amplifient artificiellement certains messages cohabitent avec les utilisateurs normaux. Détecter ces profils pose donc un vrai problème, à la fois technique et social.

Le but de ce projet, dans le cadre de l'UE IF29, est de tester et comparer deux approches d'apprentissage automatique pour identifier ces profils. La première est non supervisée : on cherche à faire ressortir des groupes naturels d'utilisateurs sans étiquette préalable. La seconde est supervisée : on définit ce qu'est un profil atypique selon nos critères, puis on entraîne un classifieur pour le reconnaître. L'idée est de voir ce que chaque approche apporte par rapport à l'autre.

Le dataset utilisé s'appelle Tweet_Worldcup. Il contient plus de 4,5 millions de tweets collectés pendant la Coupe du Monde FIFA 2018. Le choix de cet événement est intéressant : il y a beaucoup d'activité, plusieurs langues, et on sait que des comportements artificiels existent autour de ce genre de moments médiatiques (revente de billets, paris sportifs, faux comptes promotionnels).

Ce rapport décrit l'ensemble du pipeline qu'on a mis en place. On commence par l'ingestion des données brutes, puis on passe par la construction des features, le clustering, la classification supervisée et enfin la comparaison des deux modèles. Chaque choix méthodologique est expliqué au fur et à mesure.



## État de l'art

Le sujet de la détection de profils suspects sur les réseaux sociaux n'est pas nouveau. Plusieurs travaux ont marqué ce domaine et nous ont servi de base pour orienter nos choix.



### Qu'est-ce qu'un profil atypique

La notion de profil atypique peut se définir de deux manières. D'un point de vue statistique, c'est un compte dont les valeurs s'écartent fortement de la moyenne observée sur la population. D'un point de vue comportemental, c'est un compte dont les usages ne ressemblent pas à ceux d'un internaute classique. Pour ce projet, on a retenu une définition qui combine les deux : un profil est atypique quand il cumule plusieurs indicateurs extrêmes qui évoquent un comportement automatisé ou une stratégie de visibilité artificielle.



### Les travaux qui nous ont inspirés

Ferrara et al. (2016) ont publié une synthèse de référence sur les bots sociaux. Ils identifient les signaux qui permettent de distinguer un compte automatisé : ratios sociaux particuliers, fréquence de publication régulière, diversité linguistique limitée. Chu et al. (2012) ont proposé une typologie en trois catégories : humains, bots purs et comptes hybrides (cyborgs). Cette dernière catégorie est intéressante parce qu'on la retrouve souvent autour des grands événements sportifs, où des comptes officiels sont aidés par des outils de planification.

Varol et al. (2017) ont montré que même quand un compte cherche à imiter un comportement humain, certaines signatures restent détectables dans ses métadonnées. C'est un point encourageant pour notre projet, parce que cela signifie que les variables publiques contiennent vraiment de l'information utile.

Enfin, le travail de Perez et al. (2011), intitulé SPOT 1.0 et publié par l'UTT, propose une grille d'analyse en trois dimensions : agressivité, visibilité et dangerosité. L'agressivité mesure la vitesse à laquelle un compte agit, la visibilité mesure sa capacité à atteindre un large public via les mentions et hashtags, et la dangerosité évalue la part de contenu malveillant. Cette grille a directement inspiré la construction de notre score d'atypicité.



## Données et architecture technique



### Le jeu de données

Le dataset Tweet_Worldcup nous a été fourni sous forme de fichiers au format JSON Lines. Chaque ligne contient un tweet collecté via l'API Streaming de Twitter pendant la Coupe du Monde 2018. Au total, on dispose de 4 569 999 tweets, publiés par 1 843 439 utilisateurs distincts, sur une période courte qui couvre les 13 au 15 juin 2018.

Chaque tweet est un objet JSON assez dense. On y trouve le texte, les métadonnées de publication, les entités structurées (mentions, hashtags, URLs) et un objet utilisateur qui regroupe une quarantaine de champs liés au profil. C'est surtout cette dernière partie qui nous intéresse, puisque la classification porte sur les profils et pas sur les tweets.



### Choix du stockage

Pour stocker les données, on a choisi MongoDB Community Edition avec l'interface MongoDB Compass. Ce choix s'explique par la nature semi-structurée des fichiers : MongoDB accepte directement les documents JSON sans qu'on ait à définir un schéma rigide. Son moteur d'agrégation permet aussi de faire des requêtes statistiques sans avoir à charger tout en mémoire, ce qui est utile vu le volume.

L'ingestion a été faite avec pymongo, en insérant les documents par lots. On a créé une base Tweet_Worldcup contenant une seule collection nommée tweets_raw.



### Le pipeline

Le pipeline est organisé en plusieurs notebooks Jupyter qui s'enchaînent. Le premier ingère les fichiers JSON dans MongoDB. Le deuxième parcourt la collection et extrait pour chaque tweet les variables qui nous intéressent dans un fichier CSV intermédiaire. Le troisième agrège ces données au niveau utilisateur et calcule les indicateurs dérivés, ce qui produit notre fichier pivot user_twitter_data.csv. Les notebooks suivants s'occupent de la PCA, du clustering K-Means et du Random Forest.

Côté outils, on utilise Python avec les bibliothèques classiques : Pandas et NumPy pour la manipulation des données, Scikit-learn pour les modèles, Matplotlib et Seaborn pour les visualisations. L'extraction depuis MongoDB se fait par lots de 100 000 documents, écrits petit à petit sur le disque pour éviter de saturer la mémoire. Au total, l'extraction prend environ 24 minutes.



## Exploration et qualité des données

Avant de se lancer dans la modélisation, on a regardé à quoi ressemblent les données. La distribution du nombre de tweets par utilisateur est très inégale : environ deux tiers des utilisateurs n'ont publié qu'un seul tweet pendant la période d'observation, alors qu'à l'autre extrême, huit utilisateurs ont publié plus de mille tweets en trois jours. Ce dernier chiffre est déjà en soi un signal d'un usage automatisé ou très intensif.

Cette inspection a aussi révélé deux limites importantes du dataset. La première concerne les compteurs d'interaction (retweets, favoris, réponses, citations). Ils sont presque tous à zéro. La raison est simple : l'API Streaming envoie les tweets en temps réel, donc au moment où ils viennent juste d'être publiés et n'ont encore reçu aucune interaction. On ne peut donc pas s'en servir pour caractériser le comportement des utilisateurs.

La seconde limite, plus structurelle, est qu'il n'y a aucune étiquette dans les données. On ne sait pas, pour un profil donné, s'il est suspect ou pas. C'est ce qui nous a obligés à mettre en place une stratégie indirecte pour l'apprentissage supervisé.



<div style="text-align:center;"><img src="../images/DistFollowerUtilisateurs.png" alt="Figure 1 - Distribution du nombre de followers par utilisateur (echelle logarithmique)" style="max-width:80%;" /></div>

*Figure 1 - Distribution du nombre de followers par utilisateur (echelle logarithmique)*


<div style="text-align:center;"><img src="../images/BoxplotFollower.png" alt="Figure 2 - Boxplot des followers, presence d'outliers extremes" style="max-width:80%;" /></div>

*Figure 2 - Boxplot des followers, presence d'outliers extremes*


## Ingénierie des caractéristiques

La qualité des modèles dépend beaucoup des features qu'on leur donne en entrée. On a donc pris le temps de construire un ensemble de descripteurs pertinents, en s'appuyant sur la littérature et sur ce qu'on a observé pendant l'exploration.



### Stratégie d'agrégation

Le passage du niveau tweet au niveau utilisateur est l'étape centrale du pipeline. Un même utilisateur peut apparaître plusieurs fois (s'il a publié plusieurs tweets), et il a fallu décider comment combiner ses différentes lignes. On a procédé ainsi : pour les variables liées au profil (nombre d'abonnés, vérification, etc.), qui ne changent pas d'un tweet à l'autre, on garde la dernière valeur. Pour les variables de comportement (présence d'un hashtag, longueur du tweet, etc.), on calcule la moyenne sur l'ensemble des tweets de l'utilisateur. Pour les variables de comptage, on additionne.

Cette agrégation donne un fichier pivot d'une ligne par utilisateur, avec 25 colonnes.



### Les features qu'on a gardées

On a organisé les features en quatre catégories. Les features sociales décrivent la position du profil dans le réseau : nombre d'abonnés, nombre d'amis, nombre total de statuts, favoris et listes. On y ajoute des ratios qu'on a calculés nous-mêmes et qui sont plus parlants que les variables brutes. Le ratio abonnés/amis permet de repérer les profils qui suivent beaucoup de monde sans être suivis en retour, ce qui est un comportement typique des bots. Le ratio d'activité mesure le décalage entre ce que le compte produit et la taille de son audience.

Les features de contenu décrivent les habitudes éditoriales : nombre moyen de hashtags, d'URLs, de mentions, et longueur moyenne des tweets. Les features de comportement portent sur l'utilisation de la plateforme : taux de retweets, taux de citations, nombre de tweets observés dans le dataset. Enfin, les features de profil regroupent trois indicateurs binaires : statut vérifié, profil par défaut et image de profil par défaut.



### Tableau récapitulatif

On couvre bien les trois grandes dimensions demandées par la feuille de route (graphe social, contenu, comportement), avec en plus la catégorie profil qui contient des indicateurs binaires utiles.



## Construction des étiquettes par règles expertes

Pour pouvoir entraîner un modèle supervisé, il faut des étiquettes. Comme le dataset n'en contient pas, on a dû les construire nous-mêmes. La méthode qu'on a utilisée s'appelle weak supervision : on définit plusieurs règles binaires basées sur la littérature, puis on les combine en un score global qui sert de pseudo-étiquette pour l'apprentissage.



### Les critères choisis

On a retenu quatre critères. Le premier cible les profils dont le ratio abonnés/amis est inférieur à 0,05. C'est typique des comptes qui suivent beaucoup de monde sans retour. Le deuxième identifie une hyperactivité : un ratio d'activité supérieur à 20, ce qui veut dire que le compte tweete vingt fois plus qu'il n'a d'abonnés. Le troisième détecte un usage excessif de hashtags, c'est-à-dire plus de dix par tweet en moyenne. Le quatrième repère les comptes peu visibles, en fixant le seuil au premier quartile du score de visibilité (les 25 % les moins visibles).

Le score global est la somme des quatre critères, donc il varie de 0 à 4. Pour la classification, on a fait simple : un profil est étiqueté comme atypique s'il satisfait au moins deux critères. Avec ce seuil, on obtient 206 024 profils atypiques sur les 1 843 439 du dataset, soit environ 11,18 % de la population. Le déséquilibre entre les deux classes reste modéré et gérable.



### Pourquoi on ne normalise pas ici

Une question qu'on s'est posée : faut-il normaliser les variables avant d'appliquer les seuils ? La réponse est non, et c'est important de l'expliquer. Comme chaque critère est juste une comparaison binaire avec un seuil, le résultat ne dépend pas de l'échelle. Si on normalisait, il faudrait aussi standardiser le seuil, et on obtiendrait exactement le même résultat. En plus, garder les valeurs brutes rend les critères plus faciles à lire et à défendre. On normalise quand même par la suite, mais uniquement pour la PCA et le K-Means, qui sont eux sensibles aux échelles.



## Réduction de dimension par PCA

La PCA (analyse en composantes principales) sert à projeter les données sur des axes qui capturent un maximum de variance. C'est utile pour visualiser, mais aussi pour améliorer les performances de K-Means qui souffre quand il y a trop de dimensions.

On a d'abord fait une PCA classique avec StandardScaler et 2 composantes. Le résultat n'était pas convaincant : seulement 35 % de variance expliquée, et le nuage de points était totalement étiré par les profils extrêmes (les célébrités), ce qui écrasait tous les autres utilisateurs au même endroit. Difficile de tirer quoi que ce soit d'un graphique pareil.

On a donc essayé une version améliorée. D'abord, on applique un log1p sur les variables sociales (followers, friends, statuses, favourites) pour resserrer les valeurs extrêmes. Ensuite, on remplace le StandardScaler par un RobustScaler, qui utilise la médiane et l'écart interquartile au lieu de la moyenne et l'écart-type. Le RobustScaler résiste beaucoup mieux aux outliers, ce qui est exactement notre problème. Enfin, on passe à 5 composantes au lieu de 2, ce qui permet de garder plus d'information tout en gardant les deux premières pour la visualisation.



<div style="text-align:center;"><img src="../images/projectionPCA_utilisateur.png" alt="Figure 3 - Projection PCA classique des utilisateurs" style="max-width:80%;" /></div>

*Figure 3 - Projection PCA classique des utilisateurs*


<div style="text-align:center;"><img src="../images/VisuOptimisee.png" alt="Figure 4 - Visualisation optimisee apres log1p et RobustScaler" style="max-width:80%;" /></div>

*Figure 4 - Visualisation optimisee apres log1p et RobustScaler*


## Modèle non supervisé : K-Means

K-Means est un algorithme de clustering qui répartit les données en k groupes en cherchant à minimiser la distance entre chaque point et le centre de son groupe. On l'a choisi pour sa simplicité, sa rapidité sur un gros dataset comme le nôtre, et parce qu'il est facile à interpréter. On a testé deux versions : une avec PCA et une sans, pour voir ce que chacune apporte.



### Version avec PCA

Dans la première version, on applique K-Means dans l'espace réduit par la PCA. C'est beaucoup plus rapide et ça donne des graphiques plus lisibles. Pour choisir le nombre de clusters, on a utilisé la méthode du coude : on lance K-Means avec k allant de 1 à 10 et on regarde où la courbe de l'inertie présente une cassure. Ici, le coude est net à k = 4.

Les quatre clusters obtenus correspondent à quatre profils bien distincts. Le premier cluster regroupe plus d'1,18 million d'amplificateurs passifs. Ce sont des comptes ordinaires, avec environ 2 000 abonnés et un taux de retweets très élevé (93 %). Pendant la Coupe du Monde, ils consomment et relaient l'info sans rien produire eux-mêmes.

Le deuxième cluster, autour de 661 000 comptes, est composé de créateurs et partageurs actifs. Leur taille de compte est comparable au premier cluster, mais leur comportement est inversé : ils retweetent peu et partagent davantage de liens externes. Ce sont les utilisateurs qui commentent les matchs en direct et qui produisent vraiment du contenu.

Le troisième cluster ne contient que 27 comptes, mais ils sont extrêmes. En moyenne, ils ont plus de 20 millions d'abonnés et un score de visibilité immense. Ce sont les méga-célébrités et les grands médias internationaux : FIFA, joueurs vedettes, ESPN, BeIN Sports. Ce sont eux qui écrasaient le graphique au début.

Le quatrième cluster contient 326 super-diffuseurs. Moins extrêmes que les célébrités, mais avec une activité de publication très élevée pendant l'événement (en moyenne 15 tweets sur la période) et beaucoup d'URLs et de hashtags. Ce sont probablement des journalistes sportifs, des comptes officiels de clubs ou des influenceurs Twitter installés depuis longtemps.



<div style="text-align:center;"><img src="../images/coudeK.png" alt="Figure 5 - Methode du coude dans l'espace PCA, K = 4 retenu" style="max-width:80%;" /></div>

*Figure 5 - Methode du coude dans l'espace PCA, K = 4 retenu*


<div style="text-align:center;"><img src="../images/ClassificationPCA.png" alt="Figure 6 - Classification K-Means des profils dans l'espace PCA" style="max-width:80%;" /></div>

*Figure 6 - Classification K-Means des profils dans l'espace PCA*


### Version sans PCA

Dans la deuxième version, on applique K-Means directement sur les 19 variables normalisées. Le calcul est plus lourd parce qu'il faut traiter 1,8 million de points dans un espace à 19 dimensions, mais on garde toute l'information.

La méthode du coude suggère cette fois un nombre optimal de 6 clusters. C'est plus que dans la version PCA, et ça s'explique : la PCA, en concentrant la variance sur quelques axes, a tendance à fusionner des sous-groupes qui se ressemblent sur les axes principaux mais diffèrent sur des dimensions secondaires. Sans PCA, l'algorithme a la place de faire des distinctions plus fines.

Le résultat est plus détaillé. Un groupe de 332 000 créateurs de contenu engagés se distingue par un taux de retweet très bas (7 %) et une production de tweets originaux. Un deuxième groupe de 341 000 comptes correspond à des utilisateurs jeunes ou nouveaux : très peu d'abonnés (81 en moyenne), peu d'historique, mais beaucoup de relais.

Le résultat le plus intéressant, c'est un cluster de 304 000 comptes qu'on peut qualifier de spammeurs de liens et d'agrégateurs. Ils ont un taux d'URLs de 0,96 (donc presque un lien par tweet) et un taux de citations de 0,94 (94 % de quote tweets). Ce comportement est très suspect et évoque clairement de l'automatisation : ce sont sans doute des comptes qui passent leur temps à citer d'autres tweets en y greffant un lien (streaming illégal, paris sportifs, revente de billets). C'est exactement le type de profil atypique qu'on cherchait à détecter, et la version avec PCA ne le faisait pas ressortir aussi clairement.

Les autres clusters sont les super-diffuseurs professionnels (81 comptes), les amplificateurs de masse et un dernier groupe résiduel.



<div style="text-align:center;"><img src="../images/coudeSansPCA.png" alt="Figure 7 - Methode du coude sur les 19 variables normalisees, K = 6 retenu" style="max-width:80%;" /></div>

*Figure 7 - Methode du coude sur les 19 variables normalisees, K = 6 retenu*


### Que retenir des deux versions

Les deux versions ne se substituent pas l'une à l'autre, elles se complètent. La version avec PCA est plus rapide et plus lisible visuellement, mais elle simplifie trop la structure et ne sépare que les très gros profils des autres. La version sans PCA demande plus de ressources, mais elle isole un cluster spécifique de comptes manifestement automatisés que la PCA noyait dans un groupe plus large. Pour notre objectif final, qui est de détecter les profils atypiques, c'est clairement la version sans PCA qui apporte le plus.



### Limites de K-Means

K-Means a quelques défauts qu'il faut signaler. D'abord, il optimise la compacité des groupes, donc il isole les profils extrêmes dans de petits clusters, mais sans distinguer les comptes légitimes des comptes suspects. Les méga-célébrités, par exemple, ne sont pas des bots, mais elles se retrouvent dans le même genre de cluster minoritaire parce que leurs valeurs sont extrêmes. Ensuite, le choix du k reste sensible : la méthode du coude donne une indication, mais ce n'est pas une vérité absolue.



## Modèle supervisé : Random Forest

Pour l'approche supervisée, on a choisi le Random Forest. C'est un algorithme robuste, qui se débrouille bien avec des données hétérogènes et qui fournit en bonus une mesure d'importance des variables. Ce dernier point est utile pour comprendre ce qui fait que le modèle prend telle ou telle décision.



### Préparation des données

La variable cible est celle qu'on a construite plus tôt : 1 si le profil satisfait au moins deux des quatre critères du score d'atypicité, 0 sinon. Pour éviter ce qu'on appelle le data leakage (la fuite d'information), on a exclu des prédicteurs les trois variables qui ont servi à construire le label : le ratio followers/friends, le ratio d'activité et le score de visibilité. Si on les avait laissées, le modèle aurait fait du sur-apprentissage sur la définition même qu'on a posée, et les performances n'auraient plus aucun sens.

Au final, on a conservé 15 variables comme prédicteurs. On a divisé le dataset en 80 % pour l'entraînement et 20 % pour le test, avec une stratification sur la cible pour garder la même proportion d'atypiques dans les deux ensembles.



### Configuration du modèle

On a paramétré le Random Forest avec 300 arbres, une profondeur maximale de 15, un minimum de 10 échantillons pour faire un split et 5 par feuille. On a aussi mis class_weight = balanced pour que le modèle tienne compte du déséquilibre entre classes pendant l'entraînement. Ces valeurs sont un compromis entre performance et risque de surapprentissage. La parallélisation sur tous les cœurs (n_jobs=-1) accélère l'entraînement.



### Résultats

Sur l'ensemble de test, les résultats sont très bons. L'accuracy globale est de 97 %, mais comme les classes sont déséquilibrées, on regarde surtout les métriques par classe.

Sur la classe atypique, on obtient une précision de 80 %, un rappel de 100 % et un F1-score de 88 %. Le rappel parfait veut dire qu'on retrouve tous les profils étiquetés comme atypiques par les règles expertes. C'est très favorable pour un système de détection : on ne rate aucun cas. La précision de 80 % indique qu'on a quelques faux positifs : environ 10 561 profils normaux ont été signalés à tort comme atypiques.

Sur la classe normale, la précision est de 100 % et le rappel de 97 %. Seuls 117 profils atypiques (faux négatifs) sont passés à travers les mailles du filet, ce qui est très peu rapporté au volume total.



<div style="text-align:center;"><img src="../images/matriceConfusion.png" alt="Figure 8 - Matrice de confusion du Random Forest sur l'ensemble de test" style="max-width:80%;" /></div>

*Figure 8 - Matrice de confusion du Random Forest sur l'ensemble de test*


### Importance des variables

Le modèle nous donne aussi l'importance relative de chaque variable. Le résultat est assez tranché : c'est followers_count qui domine avec environ 64 % de l'importance totale. Suivent listed_count, statuses_count et friends_count. Tout en haut, on retrouve donc les variables liées à la popularité et à l'activité du compte. Ce qui confirme bien ce que la littérature nous disait.

À l'inverse, certaines variables ont eu un impact beaucoup plus faible que ce qu'on aurait pu attendre, comme le nombre de réponses ou le nombre de citations. Ce n'est pas étonnant : ces compteurs valent presque tous zéro à cause du mode de collecte par API Streaming.



<div style="text-align:center;"><img src="../images/importanceVar.png" alt="Figure 9 - Importance relative des variables du Random Forest" style="max-width:80%;" /></div>

*Figure 9 - Importance relative des variables du Random Forest*


### Validation croisée

Pour vérifier que ces performances ne dépendent pas de la façon dont on a découpé le dataset, on a fait une validation croisée à 5 plis. Le F1-score moyen est de 87,9 %, avec une variabilité limitée entre les plis. Ça confirme que le modèle est stable et que les résultats sur l'ensemble de test ne sont pas un coup de chance.



## Expérimentation et analyse comparative



### Pourquoi comparer

Maintenant qu'on a fait tourner les deux approches, l'idée est de comparer ce qu'elles donnent et de voir si elles sont d'accord, complémentaires ou contradictoires. La définition qu'on retient d'un profil atypique reste celle utilisée tout au long du projet : un utilisateur dont le comportement s'éloigne nettement de la majorité, soit par une activité inhabituelle, soit par une visibilité très faible malgré une forte production, soit par des signes d'automatisation. Cette définition est basée uniquement sur ce qu'on a dans les données. Elle ne permet pas d'affirmer qu'un compte est vraiment un bot ou un spammeur.



### Hypothèses de travail

Avant d'analyser les résultats, on avait formulé trois hypothèses. La première (H1) postulait que les profils atypiques se regrouperaient naturellement dans certains clusters quand on appliquerait K-Means. La deuxième (H2) affirmait que les variables liées à la popularité et à l'activité seraient les plus discriminantes pour identifier ces profils. La troisième (H3) anticipait que le modèle supervisé serait plus précis que le clustering pour faire de la détection automatique.



### Ce que K-Means a donné

Avec la PCA, K-Means a fait apparaître quatre groupes principaux : les amplificateurs passifs, les créateurs actifs, les méga-célébrités et les super-diffuseurs. Sans PCA, on a obtenu une segmentation plus fine avec six clusters, dont un en particulier qui regroupait des comptes au comportement très suspect (URLs systématiques et citations massives), ce qui correspond bien à un profil atypique au sens où on l'entend.

L'avantage de K-Means, c'est qu'il a fait ressortir ce groupe sans qu'on ait à le définir à l'avance. C'est typiquement le genre d'apport qu'on attend du non supervisé : il révèle des structures qu'on n'aurait pas forcément cherchées.



### Ce que Random Forest a donné

Le Random Forest, lui, a appris à reproduire notre définition à partir des étiquettes générées. Les performances sont très bonnes : 97 % d'accuracy, F1 = 88 % sur la classe atypique, rappel parfait. En clair, il retrouve presque tous les profils atypiques que notre définition désigne, avec un nombre limité d'erreurs.

L'importance des variables confirme aussi ce que dit la littérature : ce sont les variables de popularité (followers, listed) et d'activité (statuses) qui font le travail le plus important.



### Comparaison des deux approches

Les deux approches ne font pas la même chose. K-Means explore les données et révèle des groupes naturels, sans qu'on ait à dire ce qu'on cherche. Random Forest, lui, prend une définition existante (celle qu'on a posée) et apprend à la reproduire automatiquement. Le premier est utile pour comprendre et découvrir, le second est utile pour décider et automatiser.

Dans notre cas, les deux approches convergent sur un point : les profils qui se démarquent du reste de la population sont identifiables, et ce sont surtout les variables liées à la visibilité et à l'activité qui les caractérisent. Mais elles ne couvrent pas la même chose. Le K-Means sans PCA isole un cluster de spammeurs de liens que le Random Forest n'identifierait pas forcément avec les critères qu'on a choisis. À l'inverse, le Random Forest est beaucoup plus précis pour distinguer les profils atypiques au sens de notre définition métier.



### Retour sur les hypothèses

L'hypothèse H1 semble validée. K-Means a effectivement isolé plusieurs groupes au comportement très différent du reste, y compris des comptes manifestement automatisés. Ce résultat est plus net avec la version sans PCA, qui produit une segmentation plus fine.

L'hypothèse H2 est elle aussi validée. Les variables les plus importantes du Random Forest sont bien celles liées à la popularité et à l'activité (followers, listed, statuses, friends). C'est cohérent avec ce qu'on observe dans le clustering, où ce sont aussi ces dimensions qui séparent les groupes.

L'hypothèse H3 est confirmée par les chiffres. Le Random Forest atteint un F1-score de 88 % et un rappel de 100 %, ce qui est très satisfaisant pour une tâche de détection. Le non supervisé donne des résultats intéressants mais ne peut pas se substituer au supervisé pour une utilisation opérationnelle.



## Limites et perspectives

La principale limite de notre travail, c'est qu'on n'a pas de vraies étiquettes. La variable cible qu'on a construite vient d'une définition métier qu'on a posée nous-mêmes, à partir de critères inspirés de la littérature. Du coup, le Random Forest apprend en quelque sorte à reproduire nos propres règles, ce qui explique en partie pourquoi ses performances sont aussi bonnes. Il ne faut donc pas surinterpréter ces résultats : les profils détectés sont potentiellement atypiques selon nos critères, pas forcément des bots ou des spammeurs avérés.

Une autre limite, plus pratique cette fois, vient des compteurs d'interaction qui sont quasi tous à zéro à cause du mode de collecte. On n'a pas pu exploiter les retweets reçus, les favoris ou les réponses, alors que ce sont des indicateurs intéressants dans la littérature.

Pour K-Means, le résultat dépend du choix du nombre de clusters. La méthode du coude donne une indication, mais reste subjective. Une partie de l'interprétation des clusters demande aussi un regard humain : il faut comprendre ce que les chiffres veulent dire, en s'appuyant sur des connaissances du contexte (Coupe du Monde, comptes de médias, etc.).

Plusieurs pistes pourraient améliorer ce travail. Utiliser un dataset avec de vrais labels, comme Cresci 2017 ou MIB, permettrait d'évaluer les modèles sur une vérité terrain externe. Tester d'autres méthodes de clustering comme DBSCAN ou Isolation Forest serait aussi pertinent, surtout pour la détection d'anomalies. On pourrait aussi enrichir les features avec des informations temporelles (régularité des publications, distribution horaire) ou avec la structure du réseau social (qui suit qui, présence de comportements coordonnés).



## Conclusion

Ce projet nous a permis de mettre en place un pipeline complet pour la détection de profils atypiques sur Twitter, en partant de plus de 4,5 millions de tweets bruts jusqu'à la production de prédictions selon deux approches. À chaque étape, on a essayé de justifier nos choix et de garder une démarche reproductible.

Les deux approches qu'on a testées se sont révélées plus complémentaires que concurrentes. Le K-Means nous a aidés à comprendre la structure générale des données et à repérer des comportements atypiques sans avoir à les définir à l'avance. La version sans PCA a particulièrement bien marché pour isoler un cluster de comptes manifestement automatisés. Le Random Forest, lui, a permis d'automatiser la détection des profils atypiques selon notre définition, avec des performances solides (F1 = 88 %, validation croisée à 87,9 %).

Si on devait recommander une approche pour une utilisation opérationnelle, ce serait clairement le supervisé : il est plus précis, plus rapide à appliquer sur de nouvelles données et il s'appuie sur une définition explicite. Mais cette définition, justement, doit venir de quelque part. C'est là que le non supervisé est utile : il permet d'explorer les données, de repérer des comportements inattendus et de faire évoluer la définition initiale. La meilleure stratégie n'est donc pas de choisir entre les deux, mais de les utiliser ensemble.



## Bibliographie

Chu, Z., Gianvecchio, S., Wang, H., et Jajodia, S. (2012). Detecting Automation of Twitter Accounts: Are You a Human, Bot, or Cyborg? IEEE Transactions on Dependable and Secure Computing, 9(6), 811 à 824.

Ferrara, E., Varol, O., Davis, C., Menczer, F., et Flammini, A. (2016). The Rise of Social Bots. Communications of the ACM, 59(7), 96 à 104.

Perez, C., Lemercier, M., Birregah, B., et Corpel, A. (2011). SPOT 1.0: Scoring Suspicious Profiles On Twitter. Proceedings of ASONAM, 377 à 381.

Varol, O., Ferrara, E., Davis, C. A., Menczer, F., et Flammini, A. (2017). Online Human-Bot Interactions: Detection, Estimation, and Characterization. Proceedings of ICWSM, 11(1), 280 à 289.

Breiman, L. (2001). Random Forests. Machine Learning, 45(1), 5 à 32.

